# Práctica entregable  
# Clasificación de texto en redes sociales: limpieza, modelos y comparación crítica

Este notebook es el **punto de partida** de la práctica. Incluye el enunciado, los imports principales, la carga de un dataset real de redes sociales y una estructura inicial para empezar a trabajar.

El dataset por defecto será **TweetEval - Sentiment**, una tarea de clasificación de sentimiento en tweets con tres clases: `negative`, `neutral` y `positive`.

Podéis usar otro dataset de texto de redes sociales si lo justificáis adecuadamente y mantenéis una estructura de trabajo comparable.

---

# 1. Contexto y objetivo

En clase hemos trabajado análisis de sentimiento con textos relativamente largos, como reseñas de películas de IMDB. En esta práctica cambiamos de escenario: trabajaremos con textos breves procedentes de redes sociales.

Esto introduce problemas muy interesantes: menciones a usuarios, URLs, hashtags, emojis, ironía, abreviaturas, errores ortográficos, mayúsculas expresivas, repeticiones de letras, textos muy cortos y etiquetas a veces ambiguas.

El objetivo no es simplemente entrenar modelos, sino construir un **estudio comparativo serio**:

1. analizar los datos,
2. decidir cómo limpiar el texto,
3. probar varios enfoques de NLP,
4. comparar resultados,
5. analizar errores,
6. justificar qué funciona, qué no y por qué.

Como mínimo, la práctica debe incluir:

- un enfoque de **NLP clásico** con Bag of Words o TF-IDF y modelos tradicionales,
- un modelo de **red neuronal** entrenado sobre el dataset,
- y, de forma recomendada, un modelo avanzado: fine-tuning de BERT/RoBERTa/DistilBERT o uso de un LLM zero-shot/few-shot.

---

# 2. Preguntas que deben guiar el trabajo

Durante la práctica debéis responder preguntas como:

- ¿Qué ruido aparece en los textos?
- ¿Qué elementos parecen irrelevantes y cuáles podrían ser informativos?
- ¿Conviene eliminar emojis o pueden aportar información emocional?
- ¿Conviene eliminar hashtags o pueden resumir el tema del mensaje?
- ¿Qué ocurre si eliminamos menciones, URLs o signos de puntuación?
- ¿Los modelos clásicos funcionan sorprendentemente bien?
- ¿La red neuronal mejora realmente al modelo clásico o solo añade complejidad?
- ¿Un modelo preentrenado entiende mejor el lenguaje informal de redes sociales?
- ¿Un LLM zero-shot es competitivo frente a modelos entrenados específicamente?
- ¿Qué modelo recomendaríais si importan la precisión, el coste, la velocidad, la interpretabilidad o la facilidad de despliegue?

La práctica debe demostrar capacidad de **análisis, comparación y pensamiento crítico**.

---

# 3. Rúbrica resumida

| Bloque | Peso orientativo |
|---|---:|
| Comprensión del problema y análisis del dataset | 15% |
| Limpieza y preprocesamiento | 20% |
| Modelo clásico de NLP | 15% |
| Modelo de red neuronal | 15% |
| Modelo avanzado o LLM | 15% |
| Evaluación comparativa y análisis de errores | 15% |
| Claridad, reproducibilidad y comunicación | 5% |

Se valorará especialmente que las conclusiones no sean simplistas. No basta con decir “el modelo X es mejor porque tiene más accuracy”. Hay que explicar **por qué puede estar funcionando mejor**, cuándo falla, qué coste tiene y si sus ventajas compensan.

---

# 4. Instalación de dependencias

Ejecutad esta celda si estáis en un entorno nuevo, por ejemplo Colab. En local puede ser preferible instalar desde terminal o desde un entorno virtual/conda.

In [ ]:
# Si estás en Colab o en un entorno limpio, descomenta esta celda.
# !pip install -q datasets transformers evaluate accelerate emoji wordcloud

---

# 5. Imports principales

In [4]:
# %pip install datasets

In [6]:
import os
import re
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers

from datasets import load_dataset

pd.set_option("display.max_colwidth", 160)
sns.set_context("notebook")

# print("TensorFlow version:", tf.__version__)
# print("GPU disponible:", tf.config.list_physical_devices("GPU"))

---

# 6. Semilla de reproducibilidad

In [8]:
# SEED = 42

# random.seed(SEED)
# np.random.seed(SEED)
# tf.random.set_seed(SEED)
# os.environ["PYTHONHASHSEED"] = str(SEED)

---

# 7. Carga del dataset por defecto

Vamos a cargar **TweetEval - Sentiment**. Dependiendo de la versión de `datasets`, puede funcionar con alguno de estos nombres:

- `"tweet_eval", "sentiment"`
- `"cardiffnlp/tweet_eval", "sentiment"`

La celda intenta primero una opción y, si falla, prueba la otra.

In [10]:
DATASET_NAME_OPTIONS = [
    ("tweet_eval", "sentiment"),
    ("cardiffnlp/tweet_eval", "sentiment"),
]

dataset = "tweet_eval"
last_error = None

for dataset_name, config_name in DATASET_NAME_OPTIONS:
    try:
        print(f"Intentando cargar: {dataset_name} / {config_name}")
        dataset = load_dataset(dataset_name, config_name)
        print("Dataset cargado correctamente.")
        break
    except Exception as e:
        last_error = e
        print(f"No se pudo cargar {dataset_name}: {e}")

if dataset is None:
    raise RuntimeError(
        "No se pudo cargar el dataset por defecto. "
        "Comprueba la conexión a internet o instala/actualiza la librería datasets."
    ) from last_error

dataset

Intentando cargar: tweet_eval / sentiment


c:\Users\Ort\.conda\envs\reinf_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ort\.cache\huggingface\hub\datasets--tweet_eval. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 2000/2000 [00:00<00:00, 114482.74 examples/s]

Dataset cargado correctamente.


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

---

# 8. Conversión a DataFrames

Convertimos cada partición a `pandas.DataFrame` para facilitar la exploración inicial. Más adelante algunos modelos podrán trabajar directamente con arrays, `tf.data.Dataset` o datasets tokenizados de Hugging Face.

In [11]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (45615, 2)
Validation: (2000, 2)
Test: (12284, 2)


,text,label
0,"""QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin""",2
1,"""Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ""",1
2,Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.,1
3,Chase Headley's RBI double in the 8th inning off David Price snapped a Yankees streak of 33 consecutive scoreless innings against Blue Jays,1
4,"@user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017""",2


---

# 9. Normalización de etiquetas

TweetEval Sentiment usa estas etiquetas:

| ID | Etiqueta |
|---:|---|
| 0 | negative |
| 1 | neutral |
| 2 | positive |

Si usáis otro dataset, intentad adaptarlo para que tenga al menos estas columnas:

- `text`: texto original,
- `label`: etiqueta numérica,
- `label_name`: etiqueta legible.

In [12]:
required_columns = {"text", "label"}

for split_name, df in {"train": train_df, "validation": val_df, "test": test_df}.items():
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas en {split_name}: {missing}")

label_names = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

for df in [train_df, val_df, test_df]:
    df["label_name"] = df["label"].map(label_names)

train_df.sample(5)

,text,label,label_name
11449,I forgot all about Ice Cube being in the movie First Sunday. I think I seen this shit in the theaters.,0,negative
26433,playoffs are finally set. Chardon plays warren howland in the 1st round. if we win\u002c we play the winner of kenston v. tallmadge.,1,neutral
33669,Are we just going to ignore the fact that Ice Cube got his ass whoop by Ricky Smiley at the beginning of Friday After Next???,1,neutral
33013,If you live in the South Orlando area\u002c be on the lookout. @user has its 6th site opening on October 30th near the Florida Mall!,1,neutral
13399,First record of Colin Baker at the BBC: BBC2 serial The Roads to Freedom. Part 5 - shown 1 Nov 1970. #DoctorWho,1,neutral


---

# 10. Comprobaciones iniciales

Antes de modelar, conviene comprobar nulos, textos vacíos, distribución de etiquetas y duplicados.

In [14]:
def quick_dataset_checks(df, name):
    print("=" * 80)
    print(f"Split: {name}")
    print("=" * 80)
    print("Shape:", df.shape)
    print("Nulos:")
    print(df[["text", "label"]].isna().sum())
    print("Textos vacíos:", (df["text"].astype(str).str.strip() == "").sum())
    print("Distribución de etiquetas:")
    print(df["label_name"].value_counts().sort_index())
    print("Duplicados exactos de texto:", df["text"].duplicated().sum())

quick_dataset_checks(train_df, "train")
quick_dataset_checks(val_df, "validation")
quick_dataset_checks(test_df, "test")

Split: train
Shape: (45615, 3)
Nulos:
text     0
label    0
dtype: int64
Textos vacíos: 0
Distribución de etiquetas:
label_name
negative     7093
neutral     20673
positive    17849
Name: count, dtype: int64
Duplicados exactos de texto: 29
Split: validation
Shape: (2000, 3)
Nulos:
text     0
label    0
dtype: int64
Textos vacíos: 0
Distribución de etiquetas:
label_name
negative    312
neutral     869
positive    819
Name: count, dtype: int64
Duplicados exactos de texto: 0
Split: test
Shape: (12284, 3)
Nulos:
text     0
label    0
dtype: int64
Textos vacíos: 0
Distribución de etiquetas:
label_name
negative    3972
neutral     5937
positive    2375
Name: count, dtype: int64
Duplicados exactos de texto: 0


---

# 11. Distribución de clases

Si las clases están desbalanceadas, la accuracy puede ser engañosa y será especialmente importante mirar `macro-F1`.

---

# 12. Longitud de los textos

En redes sociales la longitud de los textos es muy importante: afecta a la tokenización, al padding, al vocabulario y al coste de los modelos tipo Transformer.

---

# 13. Ruido típico de redes sociales

Creamos variables sencillas para medir la presencia de URLs, menciones, hashtags, caracteres no ASCII, mayúsculas y signos expresivos.

---

# 14. Ejemplos reales por clase

Antes de limpiar o modelar, conviene leer ejemplos reales. Esto ayuda a detectar ironía, ruido, etiquetas discutibles y clases difíciles.

---

# 15. Funciones iniciales de limpieza

Esta celda no pretende resolver toda la práctica. Es un punto de partida para que podáis comparar diferentes versiones del texto:

- `text_minimal`: limpieza mínima,
- `text_moderate`: limpieza razonable para redes sociales,
- `text_aggressive`: limpieza agresiva que puede ayudar o destruir señal.

La parte importante será comprobar experimentalmente qué versión funciona mejor y por qué.

---

# 16. Variables listas para modelar

Recomendación inicial: empezar con `text_moderate` y comparar después contra `text_minimal`, `text_aggressive` y `text` original.

---

# 17. Función común de evaluación

Esta función permite evaluar cualquier modelo que devuelva predicciones de clase. Usad las mismas métricas para comparar modelos clásicos, redes neuronales, modelos preentrenados y LLMs.

---

# 18. Baseline mínimo de comprobación

Esta celda solo comprueba que los datos están listos y que el pipeline funciona. No sustituye a la batería completa de experimentos que debéis realizar.

---

# 19. Tabla de resultados

Conviene guardar todos los experimentos en una tabla final con columnas como:

- nombre del experimento,
- limpieza usada,
- representación,
- modelo,
- métricas,
- comentario interpretativo.

---

# 20. Si queréis usar otro dataset

Podéis sustituir el dataset por otro siempre que mantengáis una estructura equivalente.

Idealmente deberíais acabar teniendo tres DataFrames:

- `train_df`
- `val_df`
- `test_df`

Y cada uno debería tener, como mínimo:

- `text`
- `label`
- `label_name`

Si el dataset solo viene con train/test, podéis crear validación desde train usando `train_test_split`. Si las etiquetas vienen como texto, convertidlas a IDs numéricos.

In [ ]:
# Plantilla orientativa si usáis un CSV propio.
# No ejecutéis esta celda salvo que tengáis el archivo preparado.

# custom_df = pd.read_csv("ruta/a/mi_dataset.csv")
# custom_df = custom_df.rename(columns={
#     "nombre_columna_texto": "text",
#     "nombre_columna_etiqueta": "label_name",
# })
# label_to_id = {label: idx for idx, label in enumerate(sorted(custom_df["label_name"].unique()))}
# id_to_label = {v: k for k, v in label_to_id.items()}
# custom_df["label"] = custom_df["label_name"].map(label_to_id)
#
# train_df, temp_df = train_test_split(
#     custom_df,
#     test_size=0.30,
#     random_state=SEED,
#     stratify=custom_df["label"],
# )
# val_df, test_df = train_test_split(
#     temp_df,
#     test_size=0.50,
#     random_state=SEED,
#     stratify=temp_df["label"],
# )

---

# 21. Secciones que debéis desarrollar

A partir de aquí empieza vuestro trabajo.

## 21.1. Análisis exploratorio

Profundizad en distribución de clases, ejemplos por clase, longitud de textos, hashtags, menciones, URLs, emojis, tokens frecuentes, duplicados y posibles problemas de etiquetado.

## 21.2. Limpieza y preprocesamiento

Comparad al menos dos versiones de limpieza. No asumáis que limpiar más siempre mejora.

## 21.3. Modelos clásicos

Probad varias configuraciones: CountVectorizer, TF-IDF, unigramas, bigramas, Regresión logística, Naive Bayes y Linear SVM.

## 21.4. Red neuronal

Implementad al menos una arquitectura neuronal: LSTM, GRU, CNN 1D o Transformer block con Keras.

## 21.5. Modelo avanzado o LLM

Podéis hacer fine-tuning de DistilBERT/BERT/RoBERTa/MiniLM o evaluación zero-shot/few-shot con un LLM.

## 21.6. Comparación final y conclusiones

Incluid tabla comparativa, métricas comunes, matrices de confusión, análisis de errores y discusión crítica.

La conclusión debe responder:

> ¿Qué modelo recomendaríais para este problema concreto y bajo qué condiciones?

In [ ]:
experiment_template = pd.DataFrame([
    {
        "experiment": "ejemplo_1",
        "text_column": "text_moderate",
        "representation": "TF-IDF",
        "model": "Linear SVM",
        "accuracy": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": np.nan,
        "comment": "Sustituir por interpretación del resultado.",
    },
    {
        "experiment": "ejemplo_2",
        "text_column": "text_moderate",
        "representation": "Embedding entrenado desde cero",
        "model": "LSTM / GRU / Transformer Keras",
        "accuracy": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": np.nan,
        "comment": "Sustituir por interpretación del resultado.",
    },
    {
        "experiment": "ejemplo_3",
        "text_column": "text original",
        "representation": "Tokenizer preentrenado o prompt",
        "model": "DistilBERT / RoBERTa / LLM",
        "accuracy": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": np.nan,
        "comment": "Sustituir por interpretación del resultado.",
    },
])

experiment_template

---

# 22. Recordatorio final

Esta práctica no consiste en buscar únicamente el número más alto.

Un buen trabajo debe demostrar que entendéis:

- cómo son los datos,
- cómo afecta la limpieza,
- qué aprende cada tipo de modelo,
- qué coste tiene cada enfoque,
- qué errores comete,
- cuándo merece la pena usar un modelo más complejo,
- cuándo un modelo clásico bien trabajado puede ser suficiente.

El resultado final debe ser una comparación razonada, no una colección de modelos entrenados sin análisis.